In [12]:
import os
import random
import cv2
from glob import glob
from pathlib import Path
import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

# ====================================================================
# 1. CONFIGURACIÓN Y FILTROS DE CATEGORÍAS
# ====================================================================
n_sinteticas = 0
n_reales = 500  # <--- Ahora cambiar este número afectará correctamente al resultado

ruta_sinteticas = "./datasets/sHOMEOBJECTS_FLUXFP8"
ruta_reales = "./datasets/homeobjects-3K"
dataset_final = "./datasets/HOMEOBJECTS_LOW_MIX"

spath = ruta_sinteticas
rpath = ruta_reales
end_path = dataset_final

# ====================================================================
# 2. PROCESAMIENTO Y FILTRADO DE DATOS SINTÉTICOS
# ====================================================================
df_synth_selected = pd.DataFrame()  # DataFrame vacío por si n_sinteticas es 0

if n_sinteticas > 0:
    print("Procesando y filtrando imágenes sintéticas...")
    df = pd.read_csv(f"{spath}/prompts.csv")

    # 🛠️ CORRECCIÓN CRÍTICA: Guardamos el índice original antes de cualquier filtrado
    df["original_img_id"] = df.index

    n_total = n_sinteticas
    n_classes = df["class"].nunique()

    if n_classes == 0:
        raise ValueError("No se encontraron clases sintéticas con los criterios especificados.")

    n_per_class = n_total // n_classes

    # Muestreo balanceado por clase
    sampled_dfs = []
    for cls, group in df.groupby("class"):
        sampled_group = group.sample(n=min(n_per_class, len(group)), random_state=42)
        sampled_dfs.append(sampled_group)
    df_synth_selected = pd.concat(sampled_dfs).reset_index(drop=True)

    print("\nDistribución final de imágenes sintéticas por clase:")
    print(df_synth_selected["class"].value_counts())
else:
    print("n_sinteticas es 0. Se omitirá el procesamiento de imágenes sintéticas.")


# ====================================================================
# 3. PROCESAMIENTO Y FILTRADO DE DATOS REALES (¡SECCIÓN CORREGIDA!)
# ====================================================================
selected_pairs = []

if n_reales > 0:
    print("\nProcesando y emparejando imágenes reales...")
    
    # Asumiendo estructura estructurada de YOLO (rpath/images/train y rpath/labels/train)
    # Si tus imágenes están en la raíz de rpath, elimina la segmentación de subcarpetas
    real_images_path = os.path.join(rpath, "images", "train")
    real_labels_path = os.path.join(rpath, "labels", "train")
    
    # Validar que existan las carpetas de origen real
    if os.path.exists(real_images_path) and os.path.exists(real_labels_path):
        # Buscar imágenes JPG y PNG
        all_real_imgs = sorted(glob(os.path.join(real_images_path, "*.jpg")) + 
                               glob(os.path.join(real_images_path, "*.png")) +
                               glob(os.path.join(real_images_path, "*.jpeg")))
        
        available_pairs = []
        # Emparejar cada imagen con su respectivo .txt
        for img_path in all_real_imgs:
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(real_labels_path, f"{base_name}.txt")
            
            if os.path.exists(lbl_path):
                available_pairs.append((img_path, lbl_path))
        
        print(f"Se encontraron {len(available_pairs)} pares (imagen + etiqueta) reales disponibles en origen.")
        
        # Muestreo aleatorio de reales
        if len(available_pairs) > 0:
            random.seed(42)  # Para reproducibilidad
            sample_size = min(n_reales, len(available_pairs))
            selected_pairs = random.sample(available_pairs, k=sample_size)
            print(f"Seleccionadas exitosamente {len(selected_pairs)} imágenes reales para el dataset mezclado.")
        else:
            print("⚠️ Advertencia: No se encontraron archivos de etiquetas .txt que coincidan con las imágenes reales.")
    else:
        print(f"⚠️ Error: No se encontró la ruta de datos reales: {real_images_path} o {real_labels_path}")
else:
    print("n_reales es 0. Se omitirá el procesamiento de imágenes reales.")


# ====================================================================
# 4. CREACIÓN DE DIRECTORIOS Y ENLACES SIMBÓLICOS (YOLO FORMAT)
# ====================================================================
# Definir rutas de destino para el dataset mezclado
dest_images_dir = os.path.join(end_path, "images", "train")
dest_labels_dir = os.path.join(end_path, "labels", "train")

# Crear directorios si no existen
os.makedirs(dest_images_dir, exist_ok=True)
os.makedirs(dest_labels_dir, exist_ok=True)

print(f"\nGenerando enlaces simbólicos en: {end_path}...")

# --------------------------------------------------------------------
# A. Enlazar Imágenes Sintéticas Seleccionadas
# --------------------------------------------------------------------
if not df_synth_selected.empty:
    print("Procesando enlaces de imágenes sintéticas...")
    for idx, row in tqdm(df_synth_selected.iterrows(), total=df_synth_selected.shape[0], desc="Sintéticas"):
        orig_id = row["original_img_id"]
        
        # Construir nombres de archivos de origen
        src_img = os.path.join(spath, "images", f"{orig_id}.jpg")
        src_lbl = os.path.join(spath, "labels", f"{orig_id}.txt")
        
        # Definir nombres de destino unívocos
        dest_img = os.path.join(dest_images_dir, f"synth_{orig_id}.jpg")
        dest_lbl = os.path.join(dest_labels_dir, f"synth_{orig_id}.txt")
        
        # Crear los enlaces simbólicos
        if os.path.exists(src_img) and os.path.exists(src_lbl):
            if os.path.exists(dest_img) or os.path.islink(dest_img): os.remove(dest_img)
            if os.path.exists(dest_lbl) or os.path.islink(dest_lbl): os.remove(dest_lbl)
                
            os.symlink(os.path.abspath(src_img), dest_img)
            os.symlink(os.path.abspath(src_lbl), dest_lbl)
        else:
            print(f"⚠️ Advertencia: No se encontró el par sintético para el id {orig_id}")

# --------------------------------------------------------------------
# B. Enlazar Imágenes Reales Seleccionadas
# --------------------------------------------------------------------
if len(selected_pairs) > 0:
    print("Procesando enlaces de imágenes reales...")
    for src_img, src_lbl in tqdm(selected_pairs, desc="Reales"):
        base_name = os.path.basename(src_img)      
        base_label = os.path.basename(src_lbl)     
        
        # Definir nombres de destino unívocos (prefijo 'real_')
        dest_img = os.path.join(dest_images_dir, f"real_{base_name}")
        dest_lbl = os.path.join(dest_labels_dir, f"real_{base_label}")
        
        # Eliminar link si ya existía previamente
        if os.path.exists(dest_img) or os.path.islink(dest_img): os.remove(dest_img)
        if os.path.exists(dest_lbl) or os.path.islink(dest_lbl): os.remove(dest_lbl)
            
        os.symlink(os.path.abspath(src_img), dest_img)
        os.symlink(os.path.abspath(src_lbl), dest_lbl)

# ====================================================================
# 5. VERIFICACIÓN FINAL
# ====================================================================
total_imgs = len(glob(os.path.join(dest_images_dir, "*")))
total_lbls = len(glob(os.path.join(dest_labels_dir, "*")))

print(f"\n✅ Pipeline completado con éxito.")
print(f"📁 Directorio final: {end_path}")
print(f"🖼️ Total imágenes enlazadas: {total_imgs}")
print(f"📄 Total etiquetas enlazadas: {total_lbls}")

n_sinteticas es 0. Se omitirá el procesamiento de imágenes sintéticas.

Procesando y emparejando imágenes reales...
Se encontraron 2285 pares (imagen + etiqueta) reales disponibles en origen.
Seleccionadas exitosamente 500 imágenes reales para el dataset mezclado.

Generando enlaces simbólicos en: ./datasets/HOMEOBJECTS_LOW_MIX...
Procesando enlaces de imágenes reales...


Reales: 100%|██████████| 500/500 [00:00<00:00, 8099.68it/s]


✅ Pipeline completado con éxito.
📁 Directorio final: ./datasets/HOMEOBJECTS_LOW_MIX
🖼️ Total imágenes enlazadas: 500
📄 Total etiquetas enlazadas: 4420


class
aeroplane      250
bicycle        250
bird           250
boat           250
bottle         250
bus            250
car            250
cat            250
chair          250
cow            250
diningtable    250
dog            250
horse          250
motorbike      250
person         250
pottedplant    250
sheep          250
sofa           250
train          250
tvmonitor      250
Name: count, dtype: int64

Linking synth images and labels: 100%|██████████| 15000/15000 [00:03<00:00, 4909.68it/s]


In [ ]:
#